### T evaluation

Does M-driven timestep selection pick good `(t_start, t_end)` cells?

T has no trainable weights. It runs M over the full 11×11 grid, scalarizes (PSNR, CLIP) into `m`, takes the argmax, and applies a deviate-or-default gate.

**What we test:** ranking, regret, top-k accuracy, argmax collapse, deviate gate, sparse labels.


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "settings.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "models" / "modified-classification"
ROOT = NOTEBOOK_DIR.parents[1]
for p in (ROOT, NOTEBOOK_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from model_m import MetricPredictor
from model_t import TimestepPredictor, scalarize
from data_io import load_data, precompute_embeddings, df_to_metric_grids
from _helpers import CombinedScoreBounds, combined_score_bounds_from_arrays, resolve_device
from settings import *


/data/home/mirick/miniconda3/envs/chordedit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Inference device: None = default cuda, int = cuda:N, "cpu" = cpu.
GPU = None
DEVICE = resolve_device(GPU)
print(f"device: {DEVICE}")

def fig_title(subtitle: str) -> str:
    return f"{subtitle} for T on {METRICS_CSV.stem}, $\\delta={{{TARGET_T_DELTA}}}$"


device: cuda:4


In [4]:
"""

"""

run_dirs = sorted(d for d in OUTPUTS_DIR.iterdir() if d.is_dir())
assert run_dirs, f"No run directories in {OUTPUTS_DIR}"
RUN_DIR = run_dirs[-1]
full_df = load_data()
all_sids = sorted(full_df.sample_id.unique())
rng = np.random.default_rng(SEED)
perm = rng.permutation(all_sids)
n_holdout = max(1, round(0.2 * len(perm)))
holdout_sids = sorted(perm[:n_holdout])
holdout_df = full_df[full_df.sample_id.isin(holdout_sids)].copy()
print(f"Run: {RUN_DIR.name}  holdout: {len(holdout_sids)} images  cells: {len(holdout_df)}")

T_START_VALUES = np.sort(holdout_df.t_start.unique())
T_END_VALUES = np.sort(holdout_df.t_end.unique())
N1, N2 = len(T_START_VALUES), len(T_END_VALUES)
CELLS = N1 * N2
SAMPLE_IDS = sorted(holdout_df.sample_id.unique())
N_IMG = len(SAMPLE_IDS)
SID_TO_K = {sid: k for k, sid in enumerate(SAMPLE_IDS)}
I_OF = {v: i for i, v in enumerate(T_START_VALUES)}
J_OF = {v: j for j, v in enumerate(T_END_VALUES)}
T_START_DELTA = int(np.argmin(np.abs(T_START_VALUES - DEFAULT_T_START)))
T_END_DELTA = int(np.argmin(np.abs(T_END_VALUES - DEFAULT_T_END)))


KeyError: "['psnr', 'clip'] not in index"

In [ ]:
weights_path = RUN_DIR / "regressor_weights.pt"
ckpt = torch.load(weights_path, map_location=DEVICE, weights_only=False)
model = MetricPredictor(freeze_encoders=FREEZE_ENCODERS, device=DEVICE)
model.regressor.load_state_dict(ckpt["regressor_state_dict"])
model.regressor.set_target_stats(ckpt["target_mean"], ckpt["target_std"])
model.regressor.to(DEVICE).eval()
emb = precompute_embeddings(holdout_df.drop_duplicates("sample_id"), model, DEVICE)


In [ ]:
true_psnr, _ = df_to_metric_grids(holdout_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, PSNR_COL)
true_clip, _ = df_to_metric_grids(holdout_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, CLIP_COL)
if "combined_score_bounds" in ckpt:
    pmn, pmx, cmn, cmx = ckpt["combined_score_bounds"]
    COMBINED_BOUNDS = CombinedScoreBounds(float(pmn), float(pmx), float(cmn), float(cmx))
else:
    COMBINED_BOUNDS = combined_score_bounds_from_arrays(true_psnr, true_clip)
true_m = scalarize(true_psnr, true_clip, COMBINED_BOUNDS)

t_predictor = TimestepPredictor(
    model,
    t_start_values=T_START_VALUES,
    t_end_values=T_END_VALUES,
    scalar_stats=COMBINED_BOUNDS,
)
pred_psnr = np.zeros_like(true_psnr)
pred_clip = np.zeros_like(true_clip)
pred_m = np.zeros_like(true_m)
for sid in SAMPLE_IDS:
    k = SID_TO_K[sid]
    grid = t_predictor.predict_grid_from_emb(emb[sid]["img"], emb[sid]["src"], emb[sid]["tar"])
    pred_psnr[k], pred_clip[k], pred_m[k] = grid.psnr_grid, grid.clip_grid, grid.m_grid

print(f"default cell ({T_START_DELTA}, {T_END_DELTA}) -> t_start={T_START_VALUES[T_START_DELTA]:.3g}, t_end={T_END_VALUES[T_END_DELTA]:.3g}")
print(f"holdout grids: {true_m.shape}")


In [ ]:

"""
Per-image Spearman rho — does M rank grid cells correctly within each image?

For each image, correlate the ranked order of true vs predicted values across
all 121 cells. Reported for PSNR, CLIP, and scalarized m separately.

Interpretation:
  - median rho > 0.5   -> strong within-image ranking; T argmax is well-founded.
  - median rho 0.2-0.5 -> weak but nonzero signal; expect moderate regret.
  - median rho < 0.2   -> near-random ranking; T selection is unreliable.
  - frac rho < 0       -> fraction of images where M ranks backwards; should be ~0.
  - rho(m) matters most since T argmax operates on m, not raw PSNR/CLIP alone.
  - High rho(PSNR) but low rho(CLIP) -> check scalarization weights in settings.py.
"""

def per_image_spearman(true_grid, pred_grid):
    rhos = []
    for k in range(true_grid.shape[0]):
        rho, _ = spearmanr(true_grid[k].ravel(), pred_grid[k].ravel())
        rhos.append(np.nan if rho is None else float(rho))
    return np.array(rhos)

rho_psnr = per_image_spearman(true_psnr, pred_psnr)
rho_clip = per_image_spearman(true_clip, pred_clip)
rho_m = per_image_spearman(true_m, pred_m)
for name, r in [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]:
    rv = r[np.isfinite(r)]
    print(f"{name:5s}  median rho={np.nanmedian(r):.3f}  frac<0.2={np.mean(rv < 0.2):.2f}  frac<0={np.mean(rv < 0):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, r) in zip(axes, [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]):
    rv = r[np.isfinite(r)]
    if len(rv): ax.hist(rv, bins=min(30, max(3, len(rv))), edgecolor="white")
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"rho: {name}"); ax.set_xlabel("Spearman rho")
fig.suptitle(fig_title("Per-image Spearman"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Selection regret — how much true m do we lose by following argmax(pred_m)?

regret = true_m[oracle cell] - true_m[cell chosen by T]
Lower is better. Zero means T picked the true-best cell.

Interpretation:
  - median regret ~ 0, frac~0 high  -> T matches oracle on most images.
  - median regret < noise floor    -> gains are within label noise; defaults may suffice.
  - median regret >> noise floor   -> T is leaving real quality on the table.
  - p90 regret much larger than median -> a few images have catastrophic mis-selection.
  - Compare regret median to noise_floor_m (cell 5) to judge practical significance.
"""

def regret(true_m_grid, pred_m_grid):
    out = np.zeros(true_m_grid.shape[0])
    for k in range(true_m_grid.shape[0]):
        chosen = np.unravel_index(pred_m_grid[k].argmax(), pred_m_grid[k].shape)
        out[k] = true_m_grid[k].max() - true_m_grid[k][chosen]
    return out

reg = regret(true_m, pred_m)
print(f"regret  median={np.median(reg):.4f}  p90={np.percentile(reg, 90):.4f}  frac~0={np.mean(reg < 1e-3):.2f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(reg, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
ax.set_xlabel("regret (m units)"); ax.set_ylabel("count"); ax.set_title("selection regret")
fig.suptitle(fig_title("Selection regret"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Top-1 hit rate and top-3 overlap — does argmax(pred) land on the true best cell?

top-1 hit rate: fraction of images where pred argmax == true argmax (exact match).
top-3 overlap:  fraction of the true top-3 cells that also appear in pred top-3.

Interpretation:
  - hit rate > 0.3 on a 121-cell grid  -> better than chance (~0.008); promising.
  - hit rate > 0.5                     -> strong exact-match performance.
  - high top-3 overlap, low hit rate   -> M finds the right region but not exact peak.
  - both near 0                        -> ranking failure; check Spearman and surface plots.
  - top-3 overlap is more forgiving; useful when near-optimal cells are interchangeable.
"""

def topk_metrics(true_m_grid, pred_m_grid, k=3):
    hit1, ov = [], []
    for i in range(true_m_grid.shape[0]):
        t_order = true_m_grid[i].ravel().argsort()[::-1]
        p_order = pred_m_grid[i].ravel().argsort()[::-1]
        hit1.append(t_order[0] == p_order[0])
        ov.append(len(set(t_order[:k]) & set(p_order[:k])) / k)
    return np.mean(hit1), np.mean(ov)

h1, ov3 = topk_metrics(true_m, pred_m, k=3)
print(f"top-1 hit rate={h1:.3f}   top-3 overlap={ov3:.3f}")


### Top-N recall — is the true-best cell among T's top-N predicted outputs?

For each N, plot the percent of images whose oracle-best `(t_start, t_end)` cell appears in T's top-N cells ranked by predicted `m`.

**Interpretation**
- Steep rise in the first few N → T ranks the true best near the top quickly.
- Flat near 0 for small N → ranking failure; true best is rarely highly ranked.
- At N=1, equals top-1 hit rate; at N=121, must reach 100%.

In [ ]:
def true_best_in_top_n_pct(true_m_grid, pred_m_grid, n_values):
    """Percent of images with oracle argmax inside pred top-N (by m)."""
    n_img = true_m_grid.shape[0]
    out = []
    for n in n_values:
        hits = 0
        for i in range(n_img):
            t_order = true_m_grid[i].ravel().argsort()[::-1]
            p_order = pred_m_grid[i].ravel().argsort()[::-1]
            if t_order[0] in p_order[:n]:
                hits += 1
        out.append(100.0 * hits / n_img)
    return np.array(out)

In [ ]:
n_values = np.arange(1, CELLS + 1)
top_n_pct = true_best_in_top_n_pct(true_m, pred_m, n_values)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_values, top_n_pct, marker="o", markersize=3, linewidth=1.5)
ax.set_xlabel("True Best in Top-N Predictions")
ax.set_ylabel("Percent of Images")
ax.set_xlim(1, CELLS)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
for n_mark in (1, 3, 5, 10, 25, CELLS):
    pct = top_n_pct[n_mark - 1]
    ax.axvline(n_mark, color="k", linestyle=":", alpha=0.2)
    ax.annotate(f"N={n_mark}: {pct:.0f}%", xy=(n_mark, pct), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
fig.suptitle(fig_title("Oracle in Top-N Predictions"), fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
print(f"top-1={top_n_pct[0]:.1f}%  top-3={top_n_pct[2]:.1f}%  top-10={top_n_pct[9]:.1f}%  top-25={top_n_pct[24]:.1f}%")

In [ ]:

"""
Argmax collapse — does T pick diverse cells or always the same (t_start, t_end)?

Compare the spatial spread of true vs predicted argmax cells across images.
Red X marks the paper-default cell (t_start=0.9, t_end=0.3).

Interpretation:
  - pred spread ~ true spread, many unique pred cells -> T adapts per image (good).
  - pred spread << true spread                       -> collapse; M surface is too flat.
  - unique pred cells = 1-2 out of 121               -> T ignores image content entirely.
  - pred mass on default cell while true is spread   -> gate may be too conservative.
  - pred mass at grid edges while true is interior   -> M may overfit boundary artifacts.
"""

def argmax_cells(grid):
    return np.array([np.unravel_index(grid[k].argmax(), grid[k].shape) for k in range(grid.shape[0])])

true_arg = argmax_cells(true_m)
pred_arg = argmax_cells(pred_m)

def spread(arg):
    return arg[:, 0].std() + arg[:, 1].std()

print(f"true argmax spread = {spread(true_arg):.3f}")
print(f"pred argmax spread = {spread(pred_arg):.3f}")
print(f"unique pred cells = {len(set(map(tuple, pred_arg)))} / {CELLS}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
for ax, arg, title in zip(axes, [true_arg, pred_arg], ["true argmax", "pred argmax"]):
    H = np.zeros((N1, N2))
    for i, j in arg: H[i, j] += 1
    ax.imshow(H.T, origin="lower", cmap="Blues"); ax.set_title(title)
    ax.scatter([T_START_DELTA], [T_END_DELTA], c="r", marker="x", label="default")
    ax.set_xlabel("t_start idx"); ax.set_ylabel("t_end idx"); ax.legend(fontsize=8)
fig.suptitle(fig_title("Argmax collapse"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Label noise floor (optional) — minimum meaningful gain from re-rendering.

Requires a second CSV of the same grid cells rendered with a different seed.
noise_floor_m = median |true_m - second_seed_m| across cells.

Interpretation:
  - noise floor ~ 0.01-0.05 (typical) -> small m differences below this are not trustworthy.
  - regret median < noise floor         -> T errors are within measurement noise; acceptable.
  - regret median >> noise floor        -> real quality loss; M or T needs improvement.
  - SKIP if no second-seed data; gate and regret still usable with NOISE_FLOOR_M=0.
"""

SECOND_SEED_CSV = None
if SECOND_SEED_CSV is None or not Path(SECOND_SEED_CSV).exists():
    print("SKIP: set SECOND_SEED_CSV to compute label noise floor.")
    noise_floor_m = np.nan
else:
    second = pd.read_csv(SECOND_SEED_CSV)
    if TARGET_T_DELTA is not None: second = second[second[T_DELTA_COL] == TARGET_T_DELTA]
    second = second[second[SAMPLE_ID_COL].isin(SAMPLE_IDS)]
    s_psnr, _ = df_to_metric_grids(second, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, PSNR_COL)
    s_clip, _ = df_to_metric_grids(second, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, CLIP_COL)
    noise_floor_m = np.nanmedian(np.abs(true_m - scalarize(s_psnr, s_clip, COMBINED_BOUNDS)))
    print(f"noise floor ~ {noise_floor_m:.4f}  vs regret median {np.median(reg):.4f}")


In [ ]:

"""
Human-preferred cell rank (optional) — does the true-best m cell match human judgment?

Populate eyeball_cells = {image_index: (t_start_idx, t_end_idx)} with manually
chosen preferred cells. Reports where those cells rank under true_m.

Interpretation:
  - median rank near 0 (top of 121)  -> m metric aligns with human preference.
  - median rank >> 10                -> m may optimize the wrong objective; revisit T_TARGET_FUNC.
  - frac top-5 high                  -> even if not exact best, m finds competitive cells.
  - SKIP unless you have human annotations; automated metrics may suffice.
"""

eyeball_cells: dict[int, tuple[int, int]] = {}
if not eyeball_cells:
    print("SKIP: populate eyeball_cells with {image_index: (i, j)}.")
else:
    ranks = []
    for k, (i, j) in eyeball_cells.items():
        order = true_m[k].ravel().argsort()[::-1]
        rank = int(np.where(order == np.ravel_multi_index((i, j), (N1, N2)))[0][0])
        ranks.append(rank)
    print(f"median rank={np.median(ranks):.0f}/{CELLS}  frac top-5={np.mean(np.array(ranks) < 5):.2f}")


In [ ]:

"""
Fresh-seed realized gain (optional) — does T's pick actually improve a new render?

Re-render each image at T's chosen cell vs the default cell with a fresh seed,
then measure realized Δm. Requires wiring render_and_score() to ChordEditPipeline.

Interpretation:
  - realized Δm median > noise floor  -> T selections produce real edits, not just metric gains.
  - frac > noise_floor high           -> most flagged images genuinely improve on re-render.
  - realized Δm ~ 0                   -> M predictions don't transfer to actual renders.
  - SKIP unless you can render; surrogate-only eval (regret, Spearman) is the default path.
"""

FRESH_SEED = None

def render_and_score(image_k, t_start, t_end, seed):
    raise NotImplementedError("Wire to ChordEditPipeline")

if FRESH_SEED is None:
    print("SKIP: set FRESH_SEED and implement render_and_score().")
else:
    realized = []
    for k in range(N_IMG):
        i, j = np.unravel_index(pred_m[k].argmax(), pred_m[k].shape)
        psnr_c, clip_c = render_and_score(k, T_START_VALUES[i], T_END_VALUES[j], FRESH_SEED)
        psnr_d, clip_d = render_and_score(k, T_START_VALUES[T_START_DELTA], T_END_VALUES[T_END_DELTA], FRESH_SEED)
        realized.append(scalarize(np.array([psnr_c]), np.array([clip_c]), COMBINED_BOUNDS)[0]
                        - scalarize(np.array([psnr_d]), np.array([clip_d]), COMBINED_BOUNDS)[0])
    realized = np.array(realized)
    nf = noise_floor_m if not np.isnan(noise_floor_m) else 0.0
    print(f"realized Δm median={np.median(realized):.4f}  frac>noise_floor={np.mean(realized > nf):.2f}")


In [ ]:

"""
Deviate-or-default gate — should T change from paper defaults on this image?

T deviates only when pred_gain = max(pred_m) - pred_m[default] > noise_floor.
Compare flagged images to those truly improvable under true_m.

Interpretation:
  - precision high -> when T deviates, a better cell really exists (few false alarms).
  - recall high    -> T catches most images that truly benefit from non-default timesteps.
  - precision low  -> T deviates too often; reduce noise floor or improve M calibration.
  - recall low     -> T stays on defaults too often; lower noise floor or improve M ranking.
  - n_flagged = 0  -> gate never deviates; either defaults are optimal or M sees no gain.
  - all flagged    -> gate always deviates; noise floor may be too low or M overconfident.
"""

nf = noise_floor_m if "noise_floor_m" in dir() and not np.isnan(noise_floor_m) else NOISE_FLOOR_M
default_m = true_m[:, T_START_DELTA, T_END_DELTA]
truly_improvable = (true_m.max(axis=(1, 2)) - default_m) > nf
pred_gain = pred_m.max(axis=(1, 2)) - pred_m[:, T_START_DELTA, T_END_DELTA]
flagged = pred_gain > nf
tp = int(np.sum(flagged & truly_improvable))
precision = tp / max(int(flagged.sum()), 1)
recall = tp / max(int(truly_improvable.sum()), 1)
print(f"gate  precision={precision:.3f}  recall={recall:.3f}  ({flagged.sum()} flagged of {N_IMG})  noise_floor={nf:.4f}")


In [ ]:

"""
Error stratification — where does T fail?

Relates regret to default-cell quality and whether the true optimum sits on the
grid boundary (t_start or t_end at 0.0 or 1.0).

Interpretation:
  - regret vs default_quality correlation near 0 -> errors are not driven by baseline quality.
  - higher regret when best_at_edge=True       -> M struggles at grid boundaries; expected if
      training data is sparse near edges.
  - higher regret when best_at_edge=False      -> interior ranking failure; more concerning.
  - scatter cluster at high regret, low default -> images where defaults are poor and T also fails.
"""

df_err = pd.DataFrame({
    "sample_id": SAMPLE_IDS,
    "regret": reg,
    "default_quality": true_m[:, T_START_DELTA, T_END_DELTA],
    "best_at_edge": [(i in (0, N1-1)) or (j in (0, N2-1)) for i, j in true_arg],
})
print("regret vs default quality corr:", np.corrcoef(df_err["default_quality"], df_err["regret"])[0, 1])
print("regret by edge:", df_err.groupby("best_at_edge")["regret"].median().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(df_err["default_quality"], df_err["regret"], alpha=0.7)
axes[0].set_xlabel("default m"); axes[0].set_ylabel("regret")
axes[1].boxplot([df_err.loc[~df_err.best_at_edge, "regret"], df_err.loc[df_err.best_at_edge, "regret"]], tick_labels=["interior", "edge"])
axes[1].set_ylabel("regret")
fig.suptitle(fig_title("Error stratification"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Sparse labeling budget sweep — regret vs number of labeled cells k.

Simulates labeling only k of 121 grid cells per image, then selecting from
that subset using pred_m. Compares structured policies vs dense argmax baseline.

Policies:
  - default:    label only the paper-default cell (k=1).
  - 1d_sweeps:  ~21 cells along t_start and t_end axes through the default.
  - 5x5:        25 cells on a coarse subgrid.
  - random:     k random cells per image.

Interpretation:
  - curve near dense-argmax dashed line at low k -> M generalizes; cheap labeling suffices.
  - curve flat and high even at k=121            -> dense labels don't help; M ranking is broken.
  - 1d_sweeps beats random at same k             -> structured sweeps are worth the render cost.
  - all policies converge at k=121                 -> sanity check; should match full-grid regret.
"""

from active_label import cells_to_mask, policy_1d_sweeps, policy_structured_5x5, policy_default_only

def regret_on_mask(true_m_grid, pred_m_grid, mask):
    out = np.zeros(true_m_grid.shape[0])
    for k in range(true_m_grid.shape[0]):
        masked = np.where(mask[k], pred_m_grid[k], -np.inf)
        chosen = np.unravel_index(masked.argmax(), masked.shape)
        out[k] = true_m_grid[k].max() - true_m_grid[k][chosen]
    return out

def policy_mask(policy_name, k, rng):
    mask_3d = np.zeros((N_IMG, N1, N2), dtype=bool)
    if policy_name == "random":
        for ki in range(N_IMG):
            for idx in rng.choice(CELLS, size=min(k, CELLS), replace=False):
                i, j = np.unravel_index(idx, (N1, N2))
                mask_3d[ki, i, j] = True
        return mask_3d
    cells = policy_default_only() if policy_name == "default" else (
        policy_structured_5x5() if policy_name == "5x5" else policy_1d_sweeps())
    base = cells_to_mask(cells, T_START_VALUES, T_END_VALUES)
    return np.broadcast_to(base, (N_IMG, N1, N2)).copy()

budgets = [1, 9, 21, 25, 49, 121]
policies = ["default", "1d_sweeps", "5x5", "random"]
rng_budget = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(8, 5))
for policy_name in policies:
    xs, ys = [], []
    for k in budgets:
        if policy_name != "random" and k not in (1, 21, 25, 121): continue
        xs.append(k); ys.append(np.median(regret_on_mask(true_m, pred_m, policy_mask(policy_name, k, rng_budget))))
    ax.plot(xs, ys, marker="o", label=policy_name)
ax.axhline(np.median(reg), color="k", linestyle="--", alpha=0.5, label="dense argmax")
ax.set_xlabel("label budget k"); ax.set_ylabel("median regret"); ax.legend()
fig.suptitle(fig_title("Regret vs label budget"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
T evaluation summary — key decision metrics in one place.

Use this cell to decide whether to deploy T with the current M checkpoint:
  - Spearman m median > 0.4 AND regret median < noise floor  -> deploy with gate.
  - Argmax collapse (spread(pred) << spread(true))             -> do not deploy; fix M first.
  - Gate recall < 0.5 with many improvable images              -> lower noise floor or retrain M.
  - Sparse-label curves close to dense at k <= 25              -> active labeling is viable.
"""

print(fig_title("T summary"))
print(f"  holdout: {N_IMG} images  grid: {N1}x{N2}")
print(f"  Spearman m median: {np.nanmedian(rho_m):.3f}")
print(f"  regret median:     {np.median(reg):.4f}")
print(f"  top-1 / top-3:     {h1:.3f} / {ov3:.3f}")
print(f"  spread true/pred:  {spread(true_arg):.3f} / {spread(pred_arg):.3f}")
print(f"  gate prec/recall:  {precision:.3f} / {recall:.3f}")
